# Module 8: Advanced Topics

This notebook covers advanced topics in modern AI systems.

**Topics covered:**
- Retrieval-Augmented Generation (RAG)
- Vector similarity search
- LLM Agents and tool use
- Evaluation metrics

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from collections import defaultdict

np.random.seed(42)
%matplotlib inline

## 8.1 Text Embeddings and Similarity

In [ ]:
def cosine_similarity(a, b):
    """Compute cosine similarity between vectors."""
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-10)

def euclidean_distance(a, b):
    """Compute Euclidean distance between vectors."""
    return np.linalg.norm(a - b)

def dot_product(a, b):
    """Compute dot product between vectors."""
    return np.dot(a, b)

class SimpleEmbedder:
    """
    Simple bag-of-words embedder for demonstration.
    
    In practice, use models like sentence-transformers.
    """
    
    def __init__(self, embed_dim=64):
        self.embed_dim = embed_dim
        self.word_vectors = {}  # word -> vector
    
    def _get_word_vector(self, word):
        """Get or create word vector."""
        word = word.lower()
        if word not in self.word_vectors:
            # Random but deterministic based on word
            np.random.seed(hash(word) % 2**32)
            self.word_vectors[word] = np.random.randn(self.embed_dim)
            self.word_vectors[word] /= np.linalg.norm(self.word_vectors[word])
        return self.word_vectors[word]
    
    def embed(self, text):
        """Embed text as average of word vectors."""
        words = text.lower().split()
        if not words:
            return np.zeros(self.embed_dim)
        
        vectors = [self._get_word_vector(w) for w in words]
        avg = np.mean(vectors, axis=0)
        return avg / (np.linalg.norm(avg) + 1e-10)  # Normalize
    
    def embed_batch(self, texts):
        """Embed multiple texts."""
        return np.array([self.embed(t) for t in texts])

In [ ]:
# Test embeddings and similarity
embedder = SimpleEmbedder(embed_dim=64)

texts = [
    "The cat sat on the mat",
    "A dog lies on the rug",
    "Machine learning is fascinating",
    "Deep neural networks are powerful",
    "The weather is nice today"
]

embeddings = embedder.embed_batch(texts)
print(f"Embeddings shape: {embeddings.shape}")

# Compute pairwise similarities
print("\nCosine Similarity Matrix:")
print("(Higher = more similar)\n")

for i, t1 in enumerate(texts):
    print(f"{i}: {t1[:30]:<30}", end="  ")
    for j, t2 in enumerate(texts):
        sim = cosine_similarity(embeddings[i], embeddings[j])
        print(f"{sim:.2f}", end=" ")
    print()

## 8.2 Vector Database (Simple Implementation)

In [ ]:
class SimpleVectorDB:
    """
    Simple vector database for semantic search.
    
    In practice, use Pinecone, Weaviate, FAISS, etc.
    """
    
    def __init__(self, embedder):
        self.embedder = embedder
        self.documents = []  # Original documents
        self.embeddings = None  # Document embeddings
        self.metadata = []  # Optional metadata
    
    def add_documents(self, documents, metadata=None):
        """Add documents to the database."""
        self.documents.extend(documents)
        
        new_embeddings = self.embedder.embed_batch(documents)
        if self.embeddings is None:
            self.embeddings = new_embeddings
        else:
            self.embeddings = np.vstack([self.embeddings, new_embeddings])
        
        if metadata:
            self.metadata.extend(metadata)
        else:
            self.metadata.extend([{}] * len(documents))
    
    def search(self, query, top_k=3):
        """
        Search for most similar documents.
        
        Returns list of (document, score, metadata) tuples.
        """
        query_embedding = self.embedder.embed(query)
        
        # Compute similarities
        similarities = np.array([
            cosine_similarity(query_embedding, doc_emb)
            for doc_emb in self.embeddings
        ])
        
        # Get top-k indices
        top_indices = np.argsort(similarities)[::-1][:top_k]
        
        results = []
        for idx in top_indices:
            results.append({
                'document': self.documents[idx],
                'score': similarities[idx],
                'metadata': self.metadata[idx]
            })
        
        return results

In [ ]:
# Create a simple knowledge base
documents = [
    "The Eiffel Tower is located in Paris, France. It was constructed in 1889.",
    "Machine learning is a subset of artificial intelligence that learns from data.",
    "Python is a popular programming language for data science and AI.",
    "Neural networks are inspired by the structure of biological brains.",
    "The Great Wall of China is over 13,000 miles long.",
    "GPT models use transformer architecture for natural language processing.",
    "Deep learning requires large amounts of data and computational power.",
    "The Amazon rainforest produces about 20% of the world's oxygen.",
    "Backpropagation is the key algorithm for training neural networks.",
    "Climate change is causing rising sea levels worldwide."
]

# Create vector database
db = SimpleVectorDB(embedder)
db.add_documents(documents)

print(f"Added {len(documents)} documents to vector database")
print(f"Embedding dimension: {db.embeddings.shape[1]}")

In [ ]:
# Test semantic search
queries = [
    "What is AI?",
    "Tell me about famous landmarks",
    "How do you train neural networks?"
]

for query in queries:
    print(f"\nQuery: '{query}'")
    print("-" * 50)
    results = db.search(query, top_k=3)
    for i, r in enumerate(results):
        print(f"  {i+1}. [{r['score']:.3f}] {r['document'][:60]}...")

## 8.3 Retrieval-Augmented Generation (RAG)

In [ ]:
class SimpleRAG:
    """
    Simple RAG system.
    
    Retrieves relevant documents and augments the prompt.
    """
    
    def __init__(self, vector_db, top_k=3):
        self.vector_db = vector_db
        self.top_k = top_k
    
    def retrieve(self, query):
        """Retrieve relevant documents."""
        return self.vector_db.search(query, top_k=self.top_k)
    
    def build_prompt(self, query, retrieved_docs):
        """
        Build augmented prompt with retrieved context.
        """
        context = "\n".join([
            f"[{i+1}] {doc['document']}"
            for i, doc in enumerate(retrieved_docs)
        ])
        
        prompt = f"""Use the following context to answer the question.

Context:
{context}

Question: {query}

Answer: """
        return prompt
    
    def query(self, question):
        """
        Full RAG pipeline.
        
        In practice, the prompt would be sent to an LLM.
        """
        # Retrieve
        retrieved = self.retrieve(question)
        
        # Build prompt
        prompt = self.build_prompt(question, retrieved)
        
        return {
            'prompt': prompt,
            'retrieved_docs': retrieved,
            'query': question
        }

In [ ]:
# Test RAG system
rag = SimpleRAG(db, top_k=3)

question = "How are neural networks trained?"
result = rag.query(question)

print("RAG Output:")
print("=" * 60)
print(result['prompt'])
print("=" * 60)
print("\nRetrieved documents with scores:")
for doc in result['retrieved_docs']:
    print(f"  [{doc['score']:.3f}] {doc['document'][:50]}...")

In [ ]:
# Document chunking for RAG
def chunk_document(text, chunk_size=100, overlap=20):
    """
    Split document into overlapping chunks.
    
    Args:
        text: Document text
        chunk_size: Characters per chunk
        overlap: Overlap between chunks
    """
    chunks = []
    start = 0
    
    while start < len(text):
        end = start + chunk_size
        chunk = text[start:end]
        
        # Try to end at word boundary
        if end < len(text):
            last_space = chunk.rfind(' ')
            if last_space > chunk_size // 2:
                chunk = chunk[:last_space]
                end = start + last_space
        
        chunks.append(chunk.strip())
        start = end - overlap
    
    return chunks

# Example
long_text = """Machine learning is a branch of artificial intelligence that focuses on building 
systems that learn from data. Unlike traditional programming where rules are explicitly coded, 
machine learning algorithms discover patterns and make decisions based on training data. 
Deep learning, a subset of machine learning, uses neural networks with many layers to learn 
hierarchical representations of data."""

chunks = chunk_document(long_text, chunk_size=100, overlap=20)

print(f"Document length: {len(long_text)} chars")
print(f"Number of chunks: {len(chunks)}")
print("\nChunks:")
for i, chunk in enumerate(chunks):
    print(f"  {i+1}: [{len(chunk)} chars] {chunk[:50]}...")

## 8.4 LLM Agents and Tool Use

In [ ]:
class Tool:
    """Base class for agent tools."""
    
    def __init__(self, name, description):
        self.name = name
        self.description = description
    
    def run(self, input_str):
        raise NotImplementedError

class CalculatorTool(Tool):
    """Simple calculator tool."""
    
    def __init__(self):
        super().__init__(
            name="calculator",
            description="Performs mathematical calculations. Input should be a math expression."
        )
    
    def run(self, input_str):
        try:
            # Safe evaluation (in practice, use a proper parser)
            result = eval(input_str, {"__builtins__": {}}, 
                         {"sqrt": np.sqrt, "sin": np.sin, "cos": np.cos})
            return f"Result: {result}"
        except Exception as e:
            return f"Error: {str(e)}"

class SearchTool(Tool):
    """Mock search tool."""
    
    def __init__(self, vector_db):
        super().__init__(
            name="search",
            description="Searches the knowledge base. Input should be a search query."
        )
        self.vector_db = vector_db
    
    def run(self, input_str):
        results = self.vector_db.search(input_str, top_k=2)
        return "\n".join([f"- {r['document'][:100]}" for r in results])

class WeatherTool(Tool):
    """Mock weather tool."""
    
    def __init__(self):
        super().__init__(
            name="weather",
            description="Gets current weather for a location. Input should be a city name."
        )
    
    def run(self, input_str):
        # Mock response
        city = input_str.strip()
        temp = np.random.randint(60, 85)
        conditions = np.random.choice(["sunny", "cloudy", "rainy", "partly cloudy"])
        return f"Weather in {city}: {temp}F, {conditions}"

In [ ]:
class SimpleAgent:
    """
    Simple ReAct-style agent.
    
    In practice, the reasoning would be done by an LLM.
    This is a rule-based demonstration.
    """
    
    def __init__(self, tools):
        self.tools = {tool.name: tool for tool in tools}
    
    def get_tool_descriptions(self):
        """Get descriptions of available tools."""
        return "\n".join([
            f"- {name}: {tool.description}"
            for name, tool in self.tools.items()
        ])
    
    def parse_action(self, thought):
        """
        Simple keyword-based action parser.
        In practice, an LLM would output structured actions.
        """
        thought_lower = thought.lower()
        
        if "calculate" in thought_lower or "math" in thought_lower:
            # Extract math expression (simplified)
            import re
            numbers = re.findall(r'[\d+\-*/().]+', thought)
            if numbers:
                return "calculator", numbers[0]
        
        if "search" in thought_lower or "find" in thought_lower:
            return "search", thought
        
        if "weather" in thought_lower:
            # Try to extract city
            words = thought.split()
            for i, w in enumerate(words):
                if w.lower() == "in" and i + 1 < len(words):
                    return "weather", words[i + 1]
            return "weather", "New York"
        
        return None, None
    
    def run(self, task, max_steps=5):
        """
        Run agent on a task.
        
        Returns trace of thoughts and actions.
        """
        trace = []
        trace.append({"type": "task", "content": task})
        
        for step in range(max_steps):
            # In practice, LLM generates thought
            thought = f"Analyzing: {task}"
            trace.append({"type": "thought", "content": thought})
            
            # Parse action
            tool_name, tool_input = self.parse_action(task)
            
            if tool_name and tool_name in self.tools:
                trace.append({"type": "action", "tool": tool_name, "input": tool_input})
                
                # Execute tool
                result = self.tools[tool_name].run(tool_input)
                trace.append({"type": "observation", "content": result})
                
                # Final answer
                trace.append({"type": "answer", "content": f"Based on the tool result: {result}"})
                break
            else:
                trace.append({"type": "answer", "content": "I cannot determine which tool to use."})
                break
        
        return trace

In [ ]:
# Create agent with tools
tools = [
    CalculatorTool(),
    SearchTool(db),
    WeatherTool()
]

agent = SimpleAgent(tools)

print("Available Tools:")
print(agent.get_tool_descriptions())

In [ ]:
# Test agent with different tasks
tasks = [
    "Calculate 25 * 4 + 100",
    "Search for information about neural networks",
    "What's the weather in Paris?"
]

for task in tasks:
    print(f"\n{'='*60}")
    print(f"Task: {task}")
    print("-" * 60)
    
    trace = agent.run(task)
    for item in trace:
        if item['type'] == 'action':
            print(f"  Action: {item['tool']}({item['input']})")
        elif item['type'] == 'observation':
            print(f"  Observation: {item['content'][:100]}...")
        elif item['type'] == 'answer':
            print(f"  Answer: {item['content']}")

## 8.5 Evaluation Metrics

In [ ]:
# Retrieval Metrics

def precision_at_k(retrieved, relevant, k):
    """
    Precision@K: Fraction of top-K results that are relevant.
    """
    retrieved_k = retrieved[:k]
    relevant_in_k = len(set(retrieved_k) & set(relevant))
    return relevant_in_k / k

def recall_at_k(retrieved, relevant, k):
    """
    Recall@K: Fraction of relevant items that are in top-K.
    """
    retrieved_k = retrieved[:k]
    relevant_in_k = len(set(retrieved_k) & set(relevant))
    return relevant_in_k / len(relevant) if relevant else 0

def mrr(retrieved, relevant):
    """
    Mean Reciprocal Rank: 1 / rank of first relevant result.
    """
    for i, doc in enumerate(retrieved):
        if doc in relevant:
            return 1.0 / (i + 1)
    return 0.0

def ndcg_at_k(retrieved, relevance_scores, k):
    """
    Normalized Discounted Cumulative Gain.
    
    Args:
        retrieved: List of retrieved doc ids
        relevance_scores: Dict of doc_id -> relevance score (0-3)
        k: Number of results to consider
    """
    def dcg(scores):
        return sum([
            (2**score - 1) / np.log2(i + 2)
            for i, score in enumerate(scores)
        ])
    
    # Actual DCG
    retrieved_scores = [
        relevance_scores.get(doc, 0) for doc in retrieved[:k]
    ]
    actual_dcg = dcg(retrieved_scores)
    
    # Ideal DCG (perfect ranking)
    ideal_scores = sorted(relevance_scores.values(), reverse=True)[:k]
    ideal_dcg = dcg(ideal_scores)
    
    if ideal_dcg == 0:
        return 0.0
    return actual_dcg / ideal_dcg

In [ ]:
# Example evaluation
retrieved_docs = ['doc1', 'doc3', 'doc7', 'doc2', 'doc5']
relevant_docs = ['doc1', 'doc2', 'doc4']  # Ground truth
relevance_scores = {'doc1': 3, 'doc2': 2, 'doc3': 1, 'doc4': 3, 'doc5': 0}

print("Retrieval Evaluation:")
print(f"  Retrieved: {retrieved_docs}")
print(f"  Relevant:  {relevant_docs}")
print()

for k in [1, 3, 5]:
    p = precision_at_k(retrieved_docs, relevant_docs, k)
    r = recall_at_k(retrieved_docs, relevant_docs, k)
    print(f"  P@{k}: {p:.3f}, R@{k}: {r:.3f}")

print(f"\n  MRR: {mrr(retrieved_docs, relevant_docs):.3f}")
print(f"  NDCG@5: {ndcg_at_k(retrieved_docs, relevance_scores, 5):.3f}")

In [ ]:
# LLM Generation Metrics

def exact_match(prediction, reference):
    """Exact string match (case-insensitive)."""
    return prediction.strip().lower() == reference.strip().lower()

def word_f1(prediction, reference):
    """
    Word-level F1 score.
    
    Used for QA evaluation (e.g., SQuAD).
    """
    pred_words = set(prediction.lower().split())
    ref_words = set(reference.lower().split())
    
    if not pred_words or not ref_words:
        return float(pred_words == ref_words)
    
    common = pred_words & ref_words
    
    precision = len(common) / len(pred_words)
    recall = len(common) / len(ref_words)
    
    if precision + recall == 0:
        return 0.0
    
    return 2 * precision * recall / (precision + recall)

def bleu_1gram(prediction, reference):
    """
    Simple 1-gram BLEU score.
    
    Full BLEU uses n-grams up to 4.
    """
    pred_words = prediction.lower().split()
    ref_words = reference.lower().split()
    
    if not pred_words:
        return 0.0
    
    # Count word matches
    ref_counts = defaultdict(int)
    for word in ref_words:
        ref_counts[word] += 1
    
    matches = 0
    for word in pred_words:
        if ref_counts[word] > 0:
            matches += 1
            ref_counts[word] -= 1
    
    return matches / len(pred_words)

In [ ]:
# Example generation evaluation
test_cases = [
    ("The capital of France is Paris", "Paris is the capital of France"),
    ("Machine learning uses data", "Machine learning is a method that learns from data"),
    ("42", "42"),
    ("The answer is yes", "No, the answer is different"),
]

print("Generation Metrics:")
print("-" * 70)

for pred, ref in test_cases:
    em = exact_match(pred, ref)
    f1 = word_f1(pred, ref)
    bleu = bleu_1gram(pred, ref)
    
    print(f"\nPred: '{pred[:40]}'")
    print(f"Ref:  '{ref[:40]}'")
    print(f"  Exact Match: {em}, F1: {f1:.3f}, BLEU-1: {bleu:.3f}")

In [ ]:
# Visualize evaluation metrics comparison
metrics_data = {
    'Metric': ['Exact Match', 'F1', 'BLEU', 'ROUGE', 'BERTScore'],
    'Type': ['Exact', 'Token', 'Token', 'Token', 'Semantic'],
    'Use Case': ['QA', 'QA', 'Translation', 'Summarization', 'General'],
    'Pros': [
        'Simple, strict',
        'Partial credit',
        'N-gram precision',
        'Recall focused',
        'Semantic similarity'
    ],
    'Cons': [
        'No partial credit',
        'No word order',
        'No semantic understanding',
        'Surface level',
        'Computationally expensive'
    ]
}

print("Evaluation Metrics Overview:")
print("=" * 90)
print(f"{'Metric':<15} {'Type':<10} {'Use Case':<15} {'Pros':<25} {'Cons':<25}")
print("-" * 90)

for i in range(len(metrics_data['Metric'])):
    print(f"{metrics_data['Metric'][i]:<15} "
          f"{metrics_data['Type'][i]:<10} "
          f"{metrics_data['Use Case'][i]:<15} "
          f"{metrics_data['Pros'][i]:<25} "
          f"{metrics_data['Cons'][i]:<25}")

## 8.6 RAG Evaluation

In [ ]:
def faithfulness_check(answer, context):
    """
    Simple faithfulness check: are answer words in context?
    
    In practice, use LLM-based evaluation.
    """
    answer_words = set(answer.lower().split())
    context_words = set(context.lower().split())
    
    # Remove common words
    stopwords = {'the', 'a', 'an', 'is', 'are', 'was', 'were', 'in', 'on', 'at', 'to', 'for'}
    answer_words = answer_words - stopwords
    
    if not answer_words:
        return 1.0
    
    grounded = len(answer_words & context_words)
    return grounded / len(answer_words)

def answer_relevance(answer, question):
    """
    Check if answer is relevant to question.
    
    Simple word overlap check.
    """
    return word_f1(answer, question)

# Example RAG evaluation
question = "What is machine learning?"
context = "Machine learning is a subset of artificial intelligence that learns from data."
answer = "Machine learning is a subset of AI that learns patterns from data."

print("RAG Evaluation Example:")
print(f"  Question: {question}")
print(f"  Context:  {context}")
print(f"  Answer:   {answer}")
print()
print(f"  Faithfulness: {faithfulness_check(answer, context):.3f}")
print(f"  Relevance:    {answer_relevance(answer, question):.3f}")

## Summary

In this notebook, we covered:

1. **Text Embeddings**: Converting text to vectors for similarity search
2. **Vector Database**: Storing and retrieving documents by semantic similarity
3. **RAG**: Augmenting LLM prompts with retrieved context
4. **Agents**: LLMs that use tools to accomplish tasks
5. **Evaluation Metrics**:
   - Retrieval: P@K, R@K, MRR, NDCG
   - Generation: Exact Match, F1, BLEU
   - RAG: Faithfulness, Relevance

**Key Takeaways:**
- RAG reduces hallucination by grounding answers in retrieved documents
- Agents extend LLMs with tool use for complex tasks
- Evaluation should match the task: retrieval vs generation metrics
- Simple baselines help establish lower bounds before complex solutions